# Indexing

Goal : To add new documents to the vector DB and maintain the index.

Workflow:
1. This pipeline supports only .md and .txt files. The respective docs could be placed in a folder under `data/` (best to put in a subfolder)
2. This script can load, chunk, embed, and write them into ChromaDB.
3. Re-run any time `data/` changes. `loadDocuments()` picks up everything under `data/`, so this is safe to re-run on the full corpus.

In [1]:
import os
import sys

projectRoot = os.path.abspath(os.path.join(os.getcwd(), ".."))
srcPath = os.path.join(projectRoot, "src")
for path_ in (projectRoot, srcPath):   # projectRoot for config.py, srcPath for the pipeline modules
    if path_ not in sys.path:
        sys.path.insert(0, path_)

In [2]:
import config
import embeddings
import loader
import splitter
import vectordb

print(f"Data dir: {config.dataDir}")
print(f"Chroma dir: {config.chromaDir}")
print(f"Collection: {config.collectionName}")

D:\ProgramFiles\Anaconda\envs\mini-rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1968.48it/s]


Data dir: D:\IIIT B\PersonalProjects\Mini-RAG\mini-rag-scaffold\mini-rag\data
Chroma dir: D:\IIIT B\PersonalProjects\Mini-RAG\mini-rag-scaffold\mini-rag\db\chroma
Collection: miniRagCollection


### 1. Load documents

- By default this scans the entire `data/` folder. 
- <span style="color:red">Pass a specific subfolder to `loadDocuments()` if you only want to index newly added files.</span>

In [3]:
loadedDocuments = loader.loadDocuments()   # or loader.loadDocuments(os.path.join(config.dataDir, "new_subfolder"))
print(f"Loaded {len(loadedDocuments)} document(s)")
for document in loadedDocuments:
    print(f"  - {document.source} (id={document.documentId})")

Loaded 9 document(s)
  - 18_Pretraining_Finetuning_InstructionTuning.md (id=18_Pretraining_Finetuning_InstructionTuning)
  - 19_RLHF_DPO.md (id=19_RLHF_DPO)
  - 20_LoRA_QLoRA_PEFT.md (id=20_LoRA_QLoRA_PEFT)
  - 21_RAG.md (id=21_RAG)
  - 22_Embeddings_VectorDB.md (id=22_Embeddings_VectorDB)
  - 23_Quantization_KVCache_SpeculativeDecoding.md (id=23_Quantization_KVCache_SpeculativeDecoding)
  - 24_MixtureOfExperts.md (id=24_MixtureOfExperts)
  - transformers.md (id=transformers)
  - attention.md (id=attention)


### 2. Split into chunks

In [4]:
allChunks = splitter.splitDocuments(loadedDocuments)
print(f"{len(loadedDocuments)} document(s) -> {len(allChunks)} chunk(s)")

9 document(s) -> 87 chunk(s)


### 3. Generate embeddings

In [5]:
documentEmbeddings = embeddings.embedDocuments(allChunks)
print(f"Embeddings shape: {documentEmbeddings.shape}")

Batches: 100%|██████████| 3/3 [00:01<00:00,  1.76it/s]


Embeddings shape: (87, 384)


### 4. Write to ChromaDB

In [6]:
vectordb.add(allChunks, documentEmbeddings)
vectordb.persist()

print(f"Collection '{vectordb.collection.name}' now has {vectordb.collection.count()} chunk(s) total.")

Collection 'miniRagCollection' now has 87 chunk(s) total.


### 5. Inspecting data

Quick sanity check on what's actyally stored

In [7]:
peek = vectordb.collection.peek(limit=5)
for docId, metadata in zip(peek["ids"], peek["metadatas"]):
    print(f"{docId} -> {metadata['source']} (chunk {metadata['chunk_id']})")

transformers_0 -> transformers.md (chunk 0)
attention_0 -> attention.md (chunk 0)
18_Pretraining_Finetuning_InstructionTuning_0 -> 18_Pretraining_Finetuning_InstructionTuning.md (chunk 0)
18_Pretraining_Finetuning_InstructionTuning_1 -> 18_Pretraining_Finetuning_InstructionTuning.md (chunk 1)
18_Pretraining_Finetuning_InstructionTuning_2 -> 18_Pretraining_Finetuning_InstructionTuning.md (chunk 2)
